# Phase 2 — 슈퍼앱 리뷰 수집

**목적**: 모빌리티 슈퍼앱 경쟁사 분석 + KOKKOK Move EV 통합 전략 근거 확보

**수집 대상**
| 앱 | 패키지/ID | 분류 | 스토어 |
|---|---|---|---|
| KOKKOK Move | `com.coconutsilo.kokkokexpress.shipper` | kokkok | Google Play |
| LOCA Taxi | `com.loca.passenger` | kokkok | Google Play |
| Grab | `com.grabtaxi.passenger` | mobility_superapp | Google Play |
| Gojek | `com.gojek.app` | mobility_superapp | Google Play |
| KOKKOK Hero | `6444330521` | kokkok | App Store |

**파이프라인**
```
google-play-scraper / app-store-scraper (다국어: lo·en·th·vi)
         ↓
    DataFrame 정제 (중복·공백 제거)
         ↓
  Supabase superapp_reviews 테이블
```

> ⚠️ 충전 앱(`app_reviews`)과 **분리된 테이블**(`superapp_reviews`)에 저장합니다. 슈퍼앱은 경쟁사 분석 전용입니다.

In [ ]:
# 필요 라이브러리
# !pip install google-play-scraper app-store-scraper psycopg2-binary pandas python-dotenv

In [ ]:
import sys; sys.path.append('../')
from dotenv import load_dotenv; load_dotenv(dotenv_path='../.env')
import pandas as pd, time
from google_play_scraper import reviews, Sort, app as get_info
from app_store_scraper import AppStore
from src.db import get_connection, insert_df

## 1. 테이블 생성 (`superapp_reviews`)

> 정식 DDL은 `schema/create_tables.sql` 8번 테이블 참조. 아래는 노트북에서 바로 실행할 수 있는 동일 정의입니다.

In [ ]:
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS superapp_reviews (
                id                BIGSERIAL PRIMARY KEY,
                store             VARCHAR(20)  NOT NULL,
                app_name          VARCHAR(100) NOT NULL,
                app_category      VARCHAR(50),   -- mobility_superapp | ev_charging | kokkok
                country           CHAR(3)      NOT NULL,
                rating            SMALLINT,
                author            VARCHAR(200),
                review_date       DATE,
                content           TEXT         NOT NULL,
                lang_detected     VARCHAR(10),
                sentiment_label   VARCHAR(10),
                sentiment_score   FLOAT,
                keyword_category  VARCHAR(50),
                created_at        TIMESTAMPTZ  DEFAULT NOW()
            )
        """)
    conn.commit()
print("✅ superapp_reviews 테이블 생성")

all_rows = []

## 2. KOKKOK Move (Google Play) — 다국어 수집

In [ ]:
print("[Google Play] KOKKOK Move 수집...")
LANG_CONFIGS = [
    {'country': 'la', 'lang': 'lo'},
    {'country': 'us', 'lang': 'en'},
    {'country': 'th', 'lang': 'th'},
    {'country': 'vn', 'lang': 'vi'},
]
for cfg in LANG_CONFIGS:
    token = None
    while True:
        try:
            result, token = reviews(
                'com.coconutsilo.kokkokexpress.shipper',
                lang=cfg['lang'], country=cfg['country'],
                sort=Sort.NEWEST, count=200, continuation_token=token
            )
        except: break
        if not result: break
        for r in result:
            all_rows.append({
                'store': 'google_play', 'app_name': 'KOKKOK Move',
                'app_category': 'kokkok', 'country': 'LAO',
                'rating': r['score'], 'author': r['userName'],
                'review_date': pd.to_datetime(r['at']).date(),
                'content': r['content']
            })
        if not token: break
    print(f"  [{cfg['lang'].upper()}] {len([r for r in all_rows if r['app_name']=='KOKKOK Move'])}건")

## 3. LOCA Taxi (Google Play)

In [ ]:
print("[Google Play] LOCA Taxi 수집...")
before = len(all_rows)
for cfg in LANG_CONFIGS:
    token = None
    while True:
        try:
            result, token = reviews(
                'com.loca.passenger',
                lang=cfg['lang'], country=cfg['country'],
                sort=Sort.NEWEST, count=200, continuation_token=token
            )
        except: break
        if not result: break
        for r in result:
            all_rows.append({
                'store': 'google_play', 'app_name': 'LOCA Taxi',
                'app_category': 'kokkok', 'country': 'LAO',
                'rating': r['score'], 'author': r['userName'],
                'review_date': pd.to_datetime(r['at']).date(),
                'content': r['content']
            })
        if not token: break
print(f"  LOCA Taxi: {len(all_rows)-before}건")

## 4. Grab (동남아 샘플 — 라오·베트남·태국·영어)

In [ ]:
print("[Google Play] Grab 수집 (SEA 샘플)...")
before = len(all_rows)
GRAB_CONFIGS = [
    {'country': 'la', 'lang': 'lo'},
    {'country': 'vn', 'lang': 'vi'},
    {'country': 'th', 'lang': 'th'},
    {'country': 'us', 'lang': 'en'},
]
for cfg in GRAB_CONFIGS:
    token = None
    count = 0
    while count < 500:
        try:
            result, token = reviews(
                'com.grabtaxi.passenger',
                lang=cfg['lang'], country=cfg['country'],
                sort=Sort.NEWEST, count=200, continuation_token=token
            )
        except: break
        if not result: break
        for r in result:
            all_rows.append({
                'store': 'google_play', 'app_name': 'Grab',
                'app_category': 'mobility_superapp', 'country': 'SEA',
                'rating': r['score'], 'author': r['userName'],
                'review_date': pd.to_datetime(r['at']).date(),
                'content': r['content']
            })
            count += 1
        if not token: break
    print(f"  [{cfg['lang'].upper()}] 누적: {len(all_rows)-before}건")

## 5. Gojek (샘플)

In [ ]:
print("[Google Play] Gojek 수집 (샘플)...")
before = len(all_rows)
GOJEK_CONFIGS = [
    {'country': 'us', 'lang': 'en'},
    {'country': 'vn', 'lang': 'vi'},
    {'country': 'th', 'lang': 'th'},
]
for cfg in GOJEK_CONFIGS:
    token = None
    count = 0
    while count < 300:
        try:
            result, token = reviews(
                'com.gojek.app',
                lang=cfg['lang'], country=cfg['country'],
                sort=Sort.NEWEST, count=200, continuation_token=token
            )
        except: break
        if not result: break
        for r in result:
            all_rows.append({
                'store': 'google_play', 'app_name': 'Gojek',
                'app_category': 'mobility_superapp', 'country': 'SEA',
                'rating': r['score'], 'author': r['userName'],
                'review_date': pd.to_datetime(r['at']).date(),
                'content': r['content']
            })
            count += 1
        if not token: break
    print(f"  [{cfg['lang'].upper()}] 누적: {len(all_rows)-before}건")

## 6. KOKKOK Hero (App Store)

In [ ]:
print("[App Store] KOKKOK Hero 수집...")
before = len(all_rows)
try:
    hero_app = AppStore(country='us', app_name='KOKKOK Hero', app_id='6444330521')
    hero_app.review(how_many=500)
    for r in hero_app.reviews:
        content = r.get('review','')
        if not content: continue
        all_rows.append({
            'store': 'app_store', 'app_name': 'KOKKOK Hero',
            'app_category': 'kokkok', 'country': 'LAO',
            'rating': r.get('rating', None),
            'author': r.get('userName',''),
            'review_date': r.get('date','')[:10] if r.get('date') else None,
            'content': content
        })
    print(f"  KOKKOK Hero (App Store): {len(all_rows)-before}건")
except Exception as e:
    print(f"  ❌ KOKKOK Hero 오류: {e}")

## 7. 정제 후 Supabase 저장

In [ ]:
df = pd.DataFrame(all_rows)
df = df.drop_duplicates('content').reset_index(drop=True)
df = df[df['content'].str.strip().astype(bool)]

print(f"{'='*60}")
print(f"📊 수집 완료: 총 {len(df):,}건")
print(df.groupby(['app_name','store']).size().reset_index(name='count').to_string(index=False))

# DB 저장
insert_df(df, 'superapp_reviews')
print("\n✅ superapp_reviews 테이블 저장 완료")